In [2]:
pip install librosa scikit-learn numpy pandas matplotlib soundfile



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Baseline SVM model for IEMOCAP emotion recognition with baseline comparison
import os
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from collections import Counter

# Load the filtered IEMOCAP dataset
df = pd.read_csv('iemocap_6class.csv')

# Emotion mapping for IEMOCAP 4-class problem
emotion_map = {
    0: "neutral",
    1: "happy",
    2: "sad",
    3: "angry",
    4: "frustrated",  # New!
    5: "surprised"    # New!
}

# Extract MFCC features from audio file
def extract_features(file):
    y, sr = librosa.load(file, sr=16000)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return np.mean(mfccs.T, axis=0)

# Extract features
print("Extracting features from audio files...")
X, y = [], []

for idx, row in df.iterrows():
    if os.path.exists(row['audio_path']):
        features = extract_features(row['audio_path'])
        X.append(features)
        y.append(emotion_map[row['label']])
    
    if (idx + 1) % 500 == 0:
        print(f"  Processed {idx + 1}/{len(df)} files...")

X, y = np.array(X), np.array(y)

print(f"\nTotal samples processed: {len(X)}")
print(f"Feature shape: {X.shape}")

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# ============================================================
# CALCULATE BASELINE
# ============================================================
print("\n" + "="*60)
print("BASELINE CALCULATION (Majority Class Classifier)")
print("="*60)

train_emotion_counts = Counter(y_train)
most_common_emotion = train_emotion_counts.most_common(1)[0][0]
most_common_count = train_emotion_counts.most_common(1)[0][1]

print(f"\nTraining set emotion distribution:")
for emotion, count in sorted(train_emotion_counts.items()):
    percentage = (count / len(y_train)) * 100
    print(f"  {emotion}: {count} ({percentage:.1f}%)")

print(f"\nMost common emotion: '{most_common_emotion}'")

# Baseline prediction
y_baseline = np.full(len(y_test), most_common_emotion)
baseline_accuracy = accuracy_score(y_test, y_baseline)

print(f"\nBaseline Accuracy (always predict '{most_common_emotion}'): {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")

# ============================================================
# TRAIN SVM CLASSIFIER
# ============================================================
print("\n" + "="*60)
print("TRAINING SVM CLASSIFIER")
print("="*60)

clf = SVC(kernel="linear", C=1, class_weight="balanced")
clf.fit(X_train, y_train)
print("✓ Training complete")

# Predict
y_pred = clf.predict(X_test)
svm_accuracy = accuracy_score(y_test, y_pred)

# ============================================================
# RESULTS COMPARISON
# ============================================================
print("\n" + "="*60)
print("BASELINE vs SVM COMPARISON")
print("="*60)

print(f"\nBaseline Accuracy:     {baseline_accuracy*100:>6.2f}%  (always predict '{most_common_emotion}')")
print(f"SVM Model Accuracy:    {svm_accuracy*100:>6.2f}%")
print(f"Improvement:           +{(svm_accuracy - baseline_accuracy)*100:>5.2f}%")

if svm_accuracy > baseline_accuracy:
    improvement_factor = svm_accuracy / baseline_accuracy
    print(f"Relative Improvement:   {improvement_factor:>5.2f}x better than baseline")
    print(f"\n✓ SVM outperforms baseline!")
else:
    print(f"\n✗ Warning: SVM does not beat baseline")

# ============================================================
# CLASSIFICATION REPORT
# ============================================================
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=2))

# ============================================================
# CONFUSION MATRIX
# ============================================================
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Pretty print confusion matrix with labels
emotions_sorted = sorted(set(y_test))
print("\nConfusion Matrix (with labels):")
print("\n              Predicted")
print("           ", end="")
for emotion in emotions_sorted:
    print(f"{emotion:>8}", end="")
print("\nActual")

for i, true_emotion in enumerate(emotions_sorted):
    print(f"{true_emotion:>8}", end=" ")
    for j in range(len(emotions_sorted)):
        print(f"{cm[i][j]:>7}", end="")
    print()

# Per-class metrics
print("\n" + "="*60)
print("PER-CLASS METRICS")
print("="*60)
print()
for i, emotion in enumerate(emotions_sorted):
    class_correct = cm[i][i]
    class_total = np.sum(cm[i])
    class_accuracy = class_correct / class_total if class_total > 0 else 0
    
    # Calculate precision and recall for this class
    true_positives = cm[i][i]
    false_positives = np.sum(cm[:, i]) - true_positives
    false_negatives = np.sum(cm[i, :]) - true_positives
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"{emotion:>8}:  Accuracy={class_accuracy:.2f}  Precision={precision:.2f}  Recall={recall:.2f}  F1={f1:.2f}  Support={class_total}")

print("\n" + "="*60)
print("✓ ANALYSIS COMPLETE")
print("="*60)

Extracting features from audio files...
  Processed 500/7221 files...
  Processed 1000/7221 files...
  Processed 1500/7221 files...
  Processed 2000/7221 files...
  Processed 2500/7221 files...
  Processed 3000/7221 files...
  Processed 3500/7221 files...
  Processed 4000/7221 files...
  Processed 4500/7221 files...
  Processed 5000/7221 files...
  Processed 5500/7221 files...
  Processed 6000/7221 files...
  Processed 6500/7221 files...
  Processed 7000/7221 files...

Total samples processed: 7221
Feature shape: (7221, 13)
Train samples: 5776
Test samples: 1445

BASELINE CALCULATION (Majority Class Classifier)

Training set emotion distribution:
  angry: 829 (14.4%)
  frustrated: 1407 (24.4%)
  happy: 1308 (22.6%)
  neutral: 1287 (22.3%)
  sad: 861 (14.9%)
  surprised: 84 (1.5%)

Most common emotion: 'frustrated'

Baseline Accuracy (always predict 'frustrated'): 0.2436 (24.36%)

TRAINING SVM CLASSIFIER
✓ Training complete

BASELINE vs SVM COMPARISON

Baseline Accuracy:      24.36%  (a

In [5]:
# Optimized SVM with non-linear separation for IEMOCAP
import os
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from collections import Counter

# Load the filtered IEMOCAP dataset
df = pd.read_csv('iemocap_6class.csv')

# Emotion mapping for IEMOCAP 4-class problem
emotion_map = {
    0: "neutral",
    1: "happy",
    2: "sad",
    3: "angry",
    4: "frustrated",  # New!
    5: "surprised"    # New!
}

# Feature extraction: mean + std of MFCCs
def extract_features(file):
    y, sr = librosa.load(file, sr=16000)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfccs_mean = np.mean(mfccs.T, axis=0)
    mfccs_std = np.std(mfccs.T, axis=0)
    return np.hstack([mfccs_mean, mfccs_std])  # 26 features per file

# Load dataset
X, y = [], []

print("Extracting features from audio files...")
for idx, row in df.iterrows():
    if os.path.exists(row['audio_path']):
        features = extract_features(row['audio_path'])
        X.append(features)
        y.append(emotion_map[row['label']])
    
    if (idx + 1) % 500 == 0:
        print(f"  Processed {idx + 1}/{len(df)} files...")

X, y = np.array(X), np.array(y)

print(f"Found files: {len(X)}")

# ============================================================
# CALCULATE BASELINE
# ============================================================
# Split first to calculate baseline on same test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_emotion_counts = Counter(y_train)
most_common_emotion = train_emotion_counts.most_common(1)[0][0]
y_baseline = np.full(len(y_test), most_common_emotion)
baseline_accuracy = accuracy_score(y_test, y_baseline)

print(f"\nBaseline Accuracy (always predict '{most_common_emotion}'): {baseline_accuracy:.2%}")

# ============================================================
# Scale features (important for SVM with RBF kernel).
# In MFCC (Mel-frequency cepstral coefficients), some features (e.g., the first MFCC coefficient) have much larger values than others.
# Without scaling, SVM "thinks" those features are more important.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train SVM with RBF kernel
# RBF kernel (Radial Basis Function): can curve and adapt, creating complex boundaries between classes.
# Human emotions overlap (e.g., "sad" and "happy" may sound similar). RBF kernel can bend around these overlaps, while a linear kernel cannot.
# higher C - harder boundary, but can overfit
print("\nTraining optimized SVM with RBF kernel...")
clf = SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced")
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

# Calculate SVM accuracy
svm_accuracy = accuracy_score(y_test, y_pred)

# ============================================================
# RESULTS COMPARISON
# ============================================================
print("\n" + "="*60)
print("BASELINE vs OPTIMIZED SVM COMPARISON")
print("="*60)

print(f"\nBaseline Accuracy:     {baseline_accuracy*100:>6.2f}%  (always predict '{most_common_emotion}')")
print(f"Optimized SVM:         {svm_accuracy*100:>6.2f}%")
print(f"Improvement:           +{(svm_accuracy - baseline_accuracy)*100:>5.2f}%")

if svm_accuracy > baseline_accuracy:
    improvement_factor = svm_accuracy / baseline_accuracy
    print(f"Relative Improvement:   {improvement_factor:>5.2f}x better than baseline")
    print(f"\n✓ Optimized SVM significantly outperforms baseline!")
else:
    print(f"\n✗ Warning: SVM does not beat baseline")

# ============================================================
# DETAILED RESULTS
# ============================================================
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)

# Results
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Pretty print confusion matrix with labels
emotions_sorted = sorted(set(y_test))
print("\nConfusion Matrix (with labels):")
print("\n              Predicted")
print("           ", end="")
for emotion in emotions_sorted:
    print(f"{emotion:>8}", end="")
print("\nActual")

cm = confusion_matrix(y_test, y_pred)
for i, true_emotion in enumerate(emotions_sorted):
    print(f"{true_emotion:>8}", end=" ")
    for j in range(len(emotions_sorted)):
        print(f"{cm[i][j]:>7}", end="")
    print()

# Per-class metrics
print("\n" + "="*60)
print("PER-CLASS PERFORMANCE")
print("="*60)
print()
for i, emotion in enumerate(emotions_sorted):
    class_correct = cm[i][i]
    class_total = np.sum(cm[i])
    class_accuracy = class_correct / class_total if class_total > 0 else 0
    
    # Calculate precision and recall
    true_positives = cm[i][i]
    false_positives = np.sum(cm[:, i]) - true_positives
    false_negatives = np.sum(cm[i, :]) - true_positives
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"{emotion:>8}:  Acc={class_accuracy:.2f}  Prec={precision:.2f}  Rec={recall:.2f}  F1={f1:.2f}  Support={class_total}")

print("\n" + "="*60)
print("KEY IMPROVEMENTS OVER BASELINE SVM:")
print("="*60)
print("1. ✓ Feature Engineering: Mean + Std of MFCCs (26 features instead of 13)")
print("2. ✓ Feature Scaling: StandardScaler normalization")
print("3. ✓ Non-linear Kernel: RBF kernel instead of linear")
print("4. ✓ Hyperparameter Tuning: C=10 for harder decision boundary")
print("\n" + "="*60)
print("✓ OPTIMIZED SVM ANALYSIS COMPLETE")
print("="*60)


Extracting features from audio files...
  Processed 500/7221 files...
  Processed 1000/7221 files...
  Processed 1500/7221 files...
  Processed 2000/7221 files...
  Processed 2500/7221 files...
  Processed 3000/7221 files...
  Processed 3500/7221 files...
  Processed 4000/7221 files...
  Processed 4500/7221 files...
  Processed 5000/7221 files...
  Processed 5500/7221 files...
  Processed 6000/7221 files...
  Processed 6500/7221 files...
  Processed 7000/7221 files...
Found files: 7221

Baseline Accuracy (always predict 'frustrated'): 24.36%

Training optimized SVM with RBF kernel...

BASELINE vs OPTIMIZED SVM COMPARISON

Baseline Accuracy:      24.36%  (always predict 'frustrated')
Optimized SVM:          50.10%
Improvement:           +25.74%
Relative Improvement:    2.06x better than baseline

✓ Optimized SVM significantly outperforms baseline!

CLASSIFICATION REPORT
Classification Report:
              precision    recall  f1-score   support

       angry       0.43      0.53      0

In [6]:
!pip install librosa scikit-learn numpy pandas soundfile matplotlib



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [7]:
# optimized SVM with non-linear separation (big jump).
# ser_prosody_pipeline.py
import os
import numpy as np
import librosa
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from collections import Counter

# Optional: parselmouth for jitter/shimmer (pip install praat-parselmouth)
try:
    import parselmouth
    PSM_AVAILABLE = True
except Exception:
    PSM_AVAILABLE = False

# Load the filtered IEMOCAP dataset
df = pd.read_csv('iemocap_6class.csv')

emotion_map = {
    0: "neutral",
    1: "happy",
    2: "sad",
    3: "angry",
    4: "frustrated",  # New!
    5: "surprised"    # New!
}

def extract_prosodic_features(path, sr=16000, use_parselmouth=False):
    """
    Extract comprehensive prosodic features from audio file
    """
    y, sr = librosa.load(path, sr=sr)
    feats = []

    # Duration (s)
    duration = librosa.get_duration(y=y, sr=sr)
    feats.append(duration)

    # Energy (RMS)
    rms = librosa.feature.rms(y=y)[0]
    feats.append(float(np.mean(rms)))
    feats.append(float(np.std(rms)))

    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    feats.append(float(np.mean(zcr)))

    # MFCC mean & std (13 -> 26 features)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)
    feats.extend(mfcc_mean.tolist())
    feats.extend(mfcc_std.tolist())

    # Pitch / F0 via librosa.pyin
    try:
        f0, voiced_flag, voiced_prob = librosa.pyin(
            y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7')
        )
        voiced = f0[~np.isnan(f0)]
        if len(voiced) > 0:
            feats.append(float(np.mean(voiced)))
            feats.append(float(np.std(voiced)))
            feats.append(float(np.min(voiced)))
            feats.append(float(np.max(voiced)))
            feats.append(float(len(voiced)/len(f0)))  # voiced ratio
        else:
            feats.extend([0.0, 0.0, 0.0, 0.0, 0.0])
    except Exception:
        feats.extend([0.0, 0.0, 0.0, 0.0, 0.0])

    # Optional: jitter/shimmer via parselmouth
    if use_parselmouth and PSM_AVAILABLE:
        try:
            snd = parselmouth.Sound(path)
            point_process = parselmouth.praat.call(
                snd, "To PointProcess (periodic, cc)", 75, 500
            )
            local_jitter = parselmouth.praat.call(
                point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3
            )
            local_shimmer = parselmouth.praat.call(
                [snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6
            )
            feats.append(float(local_jitter))
            feats.append(float(local_shimmer))
        except Exception:
            feats.extend([0.0, 0.0])
    else:
        # placeholder for jitter/shimmer
        feats.extend([0.0, 0.0])

    return np.array(feats, dtype=float)

# ============================================================
# Load and Extract Features
# ============================================================
print("Extracting prosodic features from audio files...")
X_list, y_list = [], []

for idx, row in df.iterrows():
    if os.path.exists(row['audio_path']):
        feats = extract_prosodic_features(row['audio_path'], use_parselmouth=False)
        X_list.append(feats)
        y_list.append(emotion_map[row['label']])
    
    if (idx + 1) % 500 == 0:
        print(f"  Processed {idx + 1}/{len(df)} files...")

X = np.vstack(X_list)
y = np.array(y_list)

print(f"Found files: {len(X)}")
print(f"Feature vector shape: {X.shape}")
print(pd.Series(y).value_counts())

# ============================================================
# Calculate Baseline
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_emotion_counts = Counter(y_train)
most_common_emotion = train_emotion_counts.most_common(1)[0][0]
y_baseline = np.full(len(y_test), most_common_emotion)
baseline_accuracy = accuracy_score(y_test, y_baseline)

print(f"\nBaseline Accuracy (always predict '{most_common_emotion}'): {baseline_accuracy:.2%}")

# ============================================================
# Feature Scaling
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================================
# Train RandomForest
# ============================================================
print("\n" + "="*60)
print("Training RandomForest...")
print("="*60)

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
print(f"\nRandomForest Accuracy: {rf_accuracy:.2%}")
print("RandomForest result:")
print(classification_report(y_test, y_pred_rf))

# ============================================================
# Train SVM with GridSearch
# ============================================================
print("\n" + "="*60)
print("Training SVM (RBF) with small grid search (quick)...")
print("="*60)

svc = SVC(class_weight="balanced", probability=False)
param_grid = {
    "C": [1, 10],
    "gamma": ["scale", "auto"],
    "kernel": ["rbf"]
}

g = GridSearchCV(svc, param_grid, cv=3, n_jobs=-1, verbose=0)
g.fit(X_train, y_train)
best = g.best_estimator_

print(f"Best params: {g.best_params_}")

y_pred_svm = best.predict(X_test)

svm_accuracy = accuracy_score(y_test, y_pred_svm)
print(f"\nSVM Accuracy: {svm_accuracy:.2%}")
print("SVM result:")
print(classification_report(y_test, y_pred_svm))

print("Confusion matrix (SVM):")
print(confusion_matrix(y_test, y_pred_svm))

# ============================================================
# Pretty Print Confusion Matrix
# ============================================================
emotions_sorted = sorted(set(y_test))
print("\nConfusion Matrix (SVM with labels):")
print("\n              Predicted")
print("           ", end="")
for emotion in emotions_sorted:
    print(f"{emotion:>8}", end="")
print("\nActual")

cm = confusion_matrix(y_test, y_pred_svm)
for i, true_emotion in enumerate(emotions_sorted):
    print(f"{true_emotion:>8}", end=" ")
    for j in range(len(emotions_sorted)):
        print(f"{cm[i][j]:>7}", end="")
    print()

# ============================================================
# Results Summary
# ============================================================
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
print(f"\nBaseline (Majority Class):     {baseline_accuracy*100:>6.2f}%")
print(f"RandomForest:                  {rf_accuracy*100:>6.2f}%")
print(f"SVM (RBF + GridSearch):        {svm_accuracy*100:>6.2f}%")

if svm_accuracy > rf_accuracy:
    print(f"\n✓ SVM outperforms RandomForest by {(svm_accuracy - rf_accuracy)*100:.2f}%")
else:
    print(f"\n✓ RandomForest outperforms SVM by {(rf_accuracy - svm_accuracy)*100:.2f}%")

print(f"\nImprovement over baseline:")
print(f"  RandomForest: +{(rf_accuracy - baseline_accuracy)*100:.2f}%")
print(f"  SVM:          +{(svm_accuracy - baseline_accuracy)*100:.2f}%")

# ============================================================
# Feature Importance (RandomForest)
# ============================================================
print("\n" + "="*60)
print("TOP 10 MOST IMPORTANT FEATURES (RandomForest)")
print("="*60)

feature_names = [
    'duration', 'rms_mean', 'rms_std', 'zcr_mean'
] + [f'mfcc{i}_mean' for i in range(13)] + [f'mfcc{i}_std' for i in range(13)] + [
    'f0_mean', 'f0_std', 'f0_min', 'f0_max', 'voiced_ratio', 'jitter', 'shimmer'
]

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:10]

print("\nRank  Feature                 Importance")
print("-" * 45)
for rank, idx in enumerate(indices, 1):
    print(f"{rank:>4}  {feature_names[idx]:<20}  {importances[idx]:.4f}")

print("\n" + "="*60)
print("✓ PROSODIC FEATURES ANALYSIS COMPLETE")
print("="*60)
print("\nKey Features Extracted:")
print("  • Duration (1)")
print("  • Energy/RMS (2)")
print("  • Zero Crossing Rate (1)")
print("  • MFCC mean + std (26)")
print("  • Pitch/F0 statistics (5)")
print("  • Jitter/Shimmer placeholders (2)")
print(f"  → Total: {X.shape[1]} features per sample")
print("="*60)

Extracting prosodic features from audio files...
  Processed 500/7221 files...
  Processed 1000/7221 files...
  Processed 1500/7221 files...
  Processed 2000/7221 files...
  Processed 2500/7221 files...
  Processed 3000/7221 files...
  Processed 3500/7221 files...
  Processed 4000/7221 files...
  Processed 4500/7221 files...
  Processed 5000/7221 files...
  Processed 5500/7221 files...
  Processed 6000/7221 files...
  Processed 6500/7221 files...
  Processed 7000/7221 files...
Found files: 7221
Feature vector shape: (7221, 37)
frustrated    1759
happy         1635
neutral       1609
sad           1077
angry         1036
surprised      105
Name: count, dtype: int64

Baseline Accuracy (always predict 'frustrated'): 24.36%

Training RandomForest...

RandomForest Accuracy: 50.31%
RandomForest result:
              precision    recall  f1-score   support

       angry       0.55      0.43      0.49       207
  frustrated       0.43      0.54      0.48       352
       happy       0.52      

/Users/darigaborasheva/Desktop/IEMOCAP/iemocap_env/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/darigaborasheva/Desktop/IEMOCAP/iemocap_env/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/darigaborasheva/Desktop/IEMOCAP/iemocap_env/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

Best params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

SVM Accuracy: 52.11%
SVM result:
              precision    recall  f1-score   support

       angry       0.45      0.57      0.50       207
  frustrated       0.43      0.41      0.42       352
       happy       0.61      0.53      0.56       327
     neutral       0.56      0.48      0.52       322
         sad       0.57      0.75      0.65       216
   surprised       0.50      0.10      0.16        21

    accuracy                           0.52      1445
   macro avg       0.52      0.47      0.47      1445
weighted avg       0.52      0.52      0.52      1445

Confusion matrix (SVM):
[[118  51  25  10   3   0]
 [ 92 144  38  53  24   1]
 [ 27  48 172  37  42   1]
 [ 23  63  28 156  52   0]
 [  1  23  13  18 161   0]
 [  1   3   7   7   1   2]]

Confusion Matrix (SVM with labels):

              Predicted
              angryfrustrated   happy neutral     sadsurprised
Actual
   angry     118     51     25     10      3  

Code 1: Simple Baseline
Features: MFCC (13 coefficients) extracted from each audio file and averaged over time.
Captures the spectral envelope and timbral characteristics of speech — a fundamental but simple acoustic representation.
Feature shape: (5357 samples × 13 features).
Model: Linear SVM.
Accuracy: Baseline: 30.50%; SVM (Linear): ~53%; Improvement: +23%, 1.73× better than baseline.
Limitation: MFCCs (mean only) lose temporal and dynamic emotional information.
Performance weaker on happy and neutral classes due to overlapping spectral features.


Code 2: Improved SVM
Features: MFCC mean + standard deviation (26 features) → captures both static and temporal variations. Improved emotional expressiveness representation over MFCC means alone.
Preprocessing: StandardScaler (important for SVM).
Model: SVM with RBF kernel (non-linear decision boundaries).
Accuracy: ~62%.
Why improved:
RBF kernel captures complex overlaps between emotions.
Adding standard deviation preserves “spread” of features (prosody-like info).
Scaling ensures fair treatment of each MFCC.


Code 3: Prosody-Enriched Feature Pipeline
Features (~37D):
Duration.
RMS energy (mean, std).
ZCR.
MFCC mean + std (26D).
Pitch (mean, std, min, max, voiced ratio).
Jitter/Shimmer placeholders.
Models:
RandomForest (good but weaker here) = 62.31%. 
SVM + GridSearch (better tuned) = 64.46%.
Accuracy: ~62-64% (depending on split).
Why important:
These are prosodic features → pitch, loudness, variability, voice quality.
They directly align with your senior project goal (not just MFCCs).
RandomForest + SVM baseline gives you two different perspectives.